# Proyecto Hadoop: Conteo de Votos Electorales

Simulación local de elecciones en varias ciudades usando **HDFS** y **MapReduce** (Hadoop Streaming).

**Flujo:** generar CSVs ficticios → cargar en HDFS → contar votos con MapReduce → reporte final con ganador.

## 1. Diagrama de diseño del sistema

```mermaid
flowchart LR
  citySources[CityCsvSources] --> localFiles[LocalRawCsvFiles]
  localFiles --> hdfsIngest[HdfsInputPath]
  hdfsIngest --> mapPhase[MapperVoteAggregation]
  mapPhase --> shuffleSort[ShuffleSortByKey]
  shuffleSort --> reducePhase[ReducerVoteTotals]
  reducePhase --> resultFiles[HdfsResultFiles]
  resultFiles --> finalReport[FinalWinnerReport]
```

| Componente | Rol |
| --- | --- |
| Ciudades | Producen archivos CSV con votos del día |
| HDFS | Almacena todos los archivos en un directorio de entrada |
| Mapper | Lee cada línea y emite `(clave, 1)` |
| Shuffle/Sort | Agrupa por clave antes del reducer |
| Reducer | Suma votos por clave |
| Salida | Totales por candidato, por ciudad y ganador final |

## 2. Configuración y arranque de Hadoop local (Docker)

In [26]:
import csv
import random
import subprocess
from datetime import datetime, timedelta
from pathlib import Path

PROJECT_DIR = Path(".").resolve()
DATA_DIR = PROJECT_DIR / "data" / "raw_votes"
OUTPUT_DIR = PROJECT_DIR / "output"
MAPREDUCE_DIR = PROJECT_DIR / "mapreduce"

NAMENODE = "hadoop-elections-namenode"
RESOURCEMANAGER = "hadoop-elections-resourcemanager"
HDFS_INPUT = "/elections/input"
HDFS_OUTPUT_CANDIDATE = "/elections/output/candidate_totals"
HDFS_OUTPUT_CITY = "/elections/output/city_candidate_totals"
STREAMING_JAR = "/opt/hadoop-3.2.1/share/hadoop/tools/lib/hadoop-streaming-3.2.1.jar"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Directorio del proyecto: {PROJECT_DIR}")

Directorio del proyecto: /Users/franspaxi/Downloads/data_engineering_course_lab2-main/hadoop_elections


In [27]:
def run(cmd, check=True):
    """Ejecuta un comando y muestra stdout/stderr."""
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Comando falló: {' '.join(cmd)}")
    return result

def docker_exec(container, command, check=True):
    return run(["docker", "exec", container] + command, check=check)

def hdfs(args, check=True):
    return docker_exec(NAMENODE, ["hdfs", "dfs"] + args, check=check)

def yarn_streaming(input_path, output_path, mapper, reducer):
  docker_exec(
      RESOURCEMANAGER,
      [
          "yarn", "jar", STREAMING_JAR,
          "-input", input_path,
          "-output", output_path,
          "-mapper", mapper,
          "-reducer", reducer,
          "-file", f"/mapreduce/{mapper}",
          "-file", f"/mapreduce/{reducer}",
      ],
  )

In [28]:
# Arranca el mini clúster (primera vez puede tardar por descarga de imágenes)
run(["docker", "compose", "up", "-d"], check=True)

# Espera a que HDFS responda
import time
for attempt in range(30):
    result = hdfs(["-ls", "/"], check=False)
    if result.returncode == 0:
        print("HDFS listo.")
        break
    print(f"Esperando HDFS... intento {attempt + 1}/30")
    time.sleep(5)
else:
    raise RuntimeError("HDFS no respondió. Revisa: docker compose logs namenode")

$ docker compose up -d
 Container hadoop-elections-namenode Running 
 Container hadoop-elections-datanode Running 
 Container hadoop-elections-resourcemanager Running 
 Container hadoop-elections-nodemanager Running 

$ docker exec hadoop-elections-namenode hdfs dfs -ls /
Found 2 items
drwxr-xr-x   - root supergroup          0 2026-06-14 16:34 /elections
drwx------   - root supergroup          0 2026-06-14 16:10 /tmp

HDFS listo.


## 3. Generación de datos ficticios (3 ciudades)

In [29]:
CITIES = [
    {"city": "Lima", "region": "Lima", "ana_votes": 6770, "carlos_votes": 6152},
    {"city": "Arequipa", "region": "Arequipa", "ana_votes": 4796, "carlos_votes": 4357},
    {"city": "Cusco", "region": "Cusco", "ana_votes": 3668, "carlos_votes": 3332},
]
CANDIDATES = [
    {"name": "Ana Torres", "party": "Blue"},
    {"name": "Carlos Ruiz", "party": "Green"},
]
random.seed(42)
vote_id = 1
generated_files = []

for city_info in CITIES:
    file_path = DATA_DIR / f"votes_{city_info['city'].lower()}.csv"
    start_time = datetime(2026, 5, 29, 8, 0, 0)
    votes_plan = [CANDIDATES[0]] * city_info["ana_votes"] + [CANDIDATES[1]] * city_info["carlos_votes"]
    random.shuffle(votes_plan)
    with file_path.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["vote_id", "city", "region", "candidate", "party", "event_time"])
        for candidate in votes_plan:
            event_time = (start_time + timedelta(seconds=random.randint(0, 3600))).strftime("%Y-%m-%dT%H:%M:%SZ")
            writer.writerow([
                vote_id,
                city_info["city"],
                city_info["region"],
                candidate["name"],
                candidate["party"],
                event_time,
            ])
            vote_id += 1
    generated_files.append(file_path)
    city_total = city_info["ana_votes"] + city_info["carlos_votes"]
    print(f"Generado: {file_path.name} ({city_total} votos)")

print(f"Total de votos generados: {vote_id - 1}")

Generado: votes_lima.csv (12922 votos)
Generado: votes_arequipa.csv (9153 votos)
Generado: votes_cusco.csv (7000 votos)
Total de votos generados: 29075


## 4. Carga de archivos en HDFS

In [30]:
hdfs(["-rm", "-r", "-f", HDFS_INPUT, HDFS_OUTPUT_CANDIDATE, HDFS_OUTPUT_CITY], check=False)
hdfs(["-mkdir", "-p", HDFS_INPUT])

for file_path in generated_files:
    hdfs(["-put", f"/data/raw_votes/{file_path.name}", f"{HDFS_INPUT}/{file_path.name}"])

hdfs(["-ls", HDFS_INPUT])

$ docker exec hadoop-elections-namenode hdfs dfs -rm -r -f /elections/input /elections/output/candidate_totals /elections/output/city_candidate_totals
Deleted /elections/input
Deleted /elections/output/candidate_totals
Deleted /elections/output/city_candidate_totals

$ docker exec hadoop-elections-namenode hdfs dfs -mkdir -p /elections/input
$ docker exec hadoop-elections-namenode hdfs dfs -put /data/raw_votes/votes_lima.csv /elections/input/votes_lima.csv
2026-06-14 16:39:58,155 INFO sasl.SaslDataTransferClient: SASL encryption trust check: localHostTrusted = false, remoteHostTrusted = false

$ docker exec hadoop-elections-namenode hdfs dfs -put /data/raw_votes/votes_arequipa.csv /elections/input/votes_arequipa.csv
2026-06-14 16:40:00,419 INFO sasl.SaslDataTransferClient: SASL encryption trust check: localHostTrusted = false, remoteHostTrusted = false

$ docker exec hadoop-elections-namenode hdfs dfs -put /data/raw_votes/votes_cusco.csv /elections/input/votes_cusco.csv
2026-06-14 16:4

CompletedProcess(args=['docker', 'exec', 'hadoop-elections-namenode', 'hdfs', 'dfs', '-ls', '/elections/input'], returncode=0, stdout='Found 3 items\n-rw-r--r--   3 root supergroup     576248 2026-06-14 16:40 /elections/input/votes_arequipa.csv\n-rw-r--r--   3 root supergroup     398712 2026-06-14 16:40 /elections/input/votes_cusco.csv\n-rw-r--r--   3 root supergroup     699034 2026-06-14 16:39 /elections/input/votes_lima.csv\n', stderr='')

## 5. Jobs MapReduce (Hadoop Streaming)

Los scripts en `mapreduce/` usan **bash + awk** (no Python) porque las imágenes Docker de Hadoop no incluyen Python3. La lógica es la misma: el mapper emite `(clave, 1)` y el reducer suma.

In [31]:
# Job 1: votos totales por candidato
yarn_streaming(HDFS_INPUT, HDFS_OUTPUT_CANDIDATE, "mapper_candidate.sh", "reducer_count.sh")
hdfs(["-cat", f"{HDFS_OUTPUT_CANDIDATE}/part-*"])

$ docker exec hadoop-elections-resourcemanager yarn jar /opt/hadoop-3.2.1/share/hadoop/tools/lib/hadoop-streaming-3.2.1.jar -input /elections/input -output /elections/output/candidate_totals -mapper mapper_candidate.sh -reducer reducer_count.sh -file /mapreduce/mapper_candidate.sh -file /mapreduce/reducer_count.sh
packageJobJar: [/mapreduce/mapper_candidate.sh, /mapreduce/reducer_count.sh, /tmp/hadoop-unjar8216276201493639123/] [] /tmp/streamjob8912402579513295637.jar tmpDir=null

2026-06-14 16:40:07,023 WARN streaming.StreamJob: -file option is deprecated, please use generic option -files instead.
2026-06-14 16:40:07,755 INFO client.RMProxy: Connecting to ResourceManager at resourcemanager/172.18.0.4:8032
2026-06-14 16:40:07,905 INFO client.AHSProxy: Connecting to Application History server at /0.0.0.0:10200
2026-06-14 16:40:07,936 INFO client.RMProxy: Connecting to ResourceManager at resourcemanager/172.18.0.4:8032
2026-06-14 16:40:07,936 INFO client.AHSProxy: Connecting to Applicati

CompletedProcess(args=['docker', 'exec', 'hadoop-elections-namenode', 'hdfs', 'dfs', '-cat', '/elections/output/candidate_totals/part-*'], returncode=0, stdout='Ana Torres\t15234\nCarlos Ruiz\t13841\n', stderr='2026-06-14 16:40:29,504 INFO sasl.SaslDataTransferClient: SASL encryption trust check: localHostTrusted = false, remoteHostTrusted = false\n')

In [32]:
# Job 2: votos por ciudad y candidato
yarn_streaming(HDFS_INPUT, HDFS_OUTPUT_CITY, "mapper_city_candidate.sh", "reducer_count.sh")
hdfs(["-cat", f"{HDFS_OUTPUT_CITY}/part-*"])

$ docker exec hadoop-elections-resourcemanager yarn jar /opt/hadoop-3.2.1/share/hadoop/tools/lib/hadoop-streaming-3.2.1.jar -input /elections/input -output /elections/output/city_candidate_totals -mapper mapper_city_candidate.sh -reducer reducer_count.sh -file /mapreduce/mapper_city_candidate.sh -file /mapreduce/reducer_count.sh
packageJobJar: [/mapreduce/mapper_city_candidate.sh, /mapreduce/reducer_count.sh, /tmp/hadoop-unjar7878987387663345594/] [] /tmp/streamjob234792072178562362.jar tmpDir=null

2026-06-14 16:40:31,152 WARN streaming.StreamJob: -file option is deprecated, please use generic option -files instead.
2026-06-14 16:40:31,870 INFO client.RMProxy: Connecting to ResourceManager at resourcemanager/172.18.0.4:8032
2026-06-14 16:40:31,997 INFO client.AHSProxy: Connecting to Application History server at /0.0.0.0:10200
2026-06-14 16:40:32,028 INFO client.RMProxy: Connecting to ResourceManager at resourcemanager/172.18.0.4:8032
2026-06-14 16:40:32,028 INFO client.AHSProxy: Conn

CompletedProcess(args=['docker', 'exec', 'hadoop-elections-namenode', 'hdfs', 'dfs', '-cat', '/elections/output/city_candidate_totals/part-*'], returncode=0, stdout='Arequipa|Ana Torres\t4796\nArequipa|Carlos Ruiz\t4357\nCusco|Ana Torres\t3668\nCusco|Carlos Ruiz\t3332\nLima|Ana Torres\t6770\nLima|Carlos Ruiz\t6152\n', stderr='2026-06-14 16:40:54,101 INFO sasl.SaslDataTransferClient: SASL encryption trust check: localHostTrusted = false, remoteHostTrusted = false\n')

## 6. Resultado final: porcentajes y ganador

In [33]:
def read_hdfs_text(path):
    result = hdfs(["-cat", path])
    return result.stdout.strip().splitlines()

candidate_lines = read_hdfs_text(f"{HDFS_OUTPUT_CANDIDATE}/part-*")
totals = {name: int(count) for name, count in [line.split("\t") for line in candidate_lines]}
grand_total = sum(totals.values())

final_rows = []
for candidate, votes in sorted(totals.items(), key=lambda x: x[1], reverse=True):
    percentage = round(votes / grand_total * 100, 1)
    final_rows.append((candidate, votes, percentage))

winner = final_rows[0][0]
final_file = OUTPUT_DIR / "candidate_totals.csv"
with final_file.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["candidate", "total_votes", "percentage"])
    writer.writerows(final_rows)

city_lines = read_hdfs_text(f"{HDFS_OUTPUT_CITY}/part-*")
city_file = OUTPUT_DIR / "city_candidate_totals.csv"
with city_file.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["city", "candidate", "total_votes"])
    for line in city_lines:
        key, count = line.split("\t")
        city, candidate = key.split("|", 1)
        writer.writerow([city, candidate, int(count)])

summary_file = OUTPUT_DIR / "winner_summary.txt"
summary_file.write_text(
    f"Ganador: {winner}\n"
    f"Total de votos: {grand_total}\n"
    f"Resultados:\n" + "\n".join(f"  {c}: {v} ({p}%)" for c, v, p in final_rows)
)

print("=== RESULTADO FINAL ===")
print(f"Ganador: {winner}")
print(f"Total de votos: {grand_total}")
for candidate, votes, percentage in final_rows:
    print(f"{candidate}: {votes} votos ({percentage}%)")
print(f"\nArchivos guardados en: {OUTPUT_DIR}")

$ docker exec hadoop-elections-namenode hdfs dfs -cat /elections/output/candidate_totals/part-*
Ana Torres	15234
Carlos Ruiz	13841

2026-06-14 16:40:56,400 INFO sasl.SaslDataTransferClient: SASL encryption trust check: localHostTrusted = false, remoteHostTrusted = false

$ docker exec hadoop-elections-namenode hdfs dfs -cat /elections/output/city_candidate_totals/part-*
Arequipa|Ana Torres	4796
Arequipa|Carlos Ruiz	4357
Cusco|Ana Torres	3668
Cusco|Carlos Ruiz	3332
Lima|Ana Torres	6770
Lima|Carlos Ruiz	6152

2026-06-14 16:40:58,664 INFO sasl.SaslDataTransferClient: SASL encryption trust check: localHostTrusted = false, remoteHostTrusted = false

=== RESULTADO FINAL ===
Ganador: Ana Torres
Total de votos: 29075
Ana Torres: 15234 votos (52.4%)
Carlos Ruiz: 13841 votos (47.6%)

Archivos guardados en: /Users/franspaxi/Downloads/data_engineering_course_lab2-main/hadoop_elections/output


## 7. Preguntas de discusión

**¿Por qué HDFS divide los archivos en bloques?**
Para distribuir datos en varios nodos, paralelizar lecturas y tolerar fallos con réplicas. Un archivo grande se procesa en paralelo por varios mappers.

**¿De qué es responsable el mapper?**
Leer registros de entrada y emitir pares `(clave, valor)` — aquí cada voto se convierte en `(candidato, 1)` o `(ciudad|candidato, 1)`.

**¿De qué es responsable el reducer?**
Recibir todos los valores de una misma clave (tras shuffle/sort) y agregarlos — en este proyecto, sumar votos.

**¿Cuándo sería Hadoop una mejor opción que un script normal de Python?**
Cuando los datos son demasiado grandes para una sola máquina, cuando necesitas procesamiento batch distribuido y tolerancia a fallos, o cuando el volumen crece y quieres escalar agregando nodos en lugar de reescribir todo.